# Recomendación de títulos de Netflix: Modelado

Encontrar películas y series similares por descripción y metadatos.

TF-IDF de descripción, categorías, reparto y director; vecinos por coseno. Baseline por año de estreno.

Consulta la [metodología](../docs/metodologia.md). Los reportes corresponden a la última ejecución de cada etapa.


In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import Markdown, display
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("netflix_recommender")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre el notebook desde el proyecto, notebooks o su carpeta contenedora.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
for folder in ("data/raw", "data/intermediate", "data/processed", "data/output", "models", "reports"):
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
from netflix_recommender.alerts import configurar_notebook, mostrar_alertas
configurar_notebook()


In [ ]:
from netflix_recommender.flow import run_stage
metricas, comentario = run_stage(PROJECT_ROOT, "modelado")
display(Markdown(comentario))

from netflix_recommender.content import recommend
QUERY_ID = None  # None usa el primer ID del catálogo; puedes reemplazarlo.
TOP_N = 10
display(recommend(PROJECT_ROOT, item_id=QUERY_ID, top_n=TOP_N))


## Alertas

Avisos agrupados con descripción y acción sugerida. Los errores no se ocultan; no registrar warnings no garantiza que el análisis sea válido.

In [ ]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/04_modelado.json")
from netflix_recommender.informe import generar_informe
generar_informe(PROJECT_ROOT)
